# Week 2, Day 2: Indexing, slicing and masks: selecting trials and neurons

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w002/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
# 20 neurons, 60 trials, 200 bins of 5 ms (1 second per trial)
data = rng.poisson(0.1, size=(20, 60, 200))

print(data[3].shape)            # one neuron, all trials and bins: (60, 200)
print(data[:, 0].shape)         # all neurons, first trial: (20, 200)
print(data[:5, :, 100:].shape)  # first 5 neurons, all trials, second half of each trial
print(data[..., -1].shape)      # '...' means 'all the other axes': the last bin of everything
print(data[:, ::2].shape)       # every other trial

In [ ]:
# Trial information, one value per trial
contrast = rng.choice([0.0, 0.25, 0.5, 1.0], size=60)
correct = rng.random(60) < 0.7
area = np.array(["V1"] * 8 + ["MOs"] * 7 + ["CA1"] * 5)   # one label per neuron

high = contrast >= 0.5
print("high-contrast trials:", high.sum())
print("high and correct:", (high & correct).sum())      # & is 'and', | is 'or', ~ is 'not'

v1 = area == "V1"
v1_high = data[v1][:, high & correct]                    # V1 neurons, high-contrast correct trials
print("V1, high, correct:", v1_high.shape)

In [ ]:
mean_rate = data.mean(axis=(1, 2)) / 0.005          # Hz, one per neuron
order = np.argsort(mean_rate)[::-1]                 # neuron indices, fastest first
print("fastest 3 neurons:", order[:3], np.round(mean_rate[order[:3]], 1))
sorted_data = data[order]                           # neurons reordered by rate
print(sorted_data.shape)

In [ ]:
x = np.arange(10)
v = x[2:5]          # view
v[:] = -1
print("after changing the slice:", x)

y = np.arange(10)
c = y[y > 6]        # copy (mask)
c[:] = -1
print("after changing the masked copy:", y)

w = np.arange(10)[2:5].copy()   # an explicit copy is always safe
print("np.shares_memory(x, v):", np.shares_memory(x, v))

### Exercise: Response in a window

Using `data` (5 ms bins), compute `resp`: each neuron's spike count on each trial between 250 ms and 500 ms after trial start.

In [ ]:
resp = ...
print(resp.shape)

In [ ]:
import numpy as np
def test_shape():
    "resp has one value per neuron and trial"
    assert resp.shape == (20, 60)

def test_values():
    "resp is the spike count from 250 to 500 ms"
    assert np.array_equal(resp, data[:, :, 50:100].sum(axis=2))

test_shape()
test_values()
print("All checks passed")

### Exercise: Compare conditions

For neurons in `MOs` only, compute `diff`: the mean of `resp` on trials with contrast at least 0.5, minus the mean on trials with zero contrast.

In [ ]:
mos = ...
diff = ...
print("difference in spikes:", diff)

In [ ]:
import numpy as np
def test_diff():
    "diff is the mean difference between high- and zero-contrast trials for MOs neurons"
    m = area == "MOs"
    expect = resp[m][:, contrast >= 0.5].mean() - resp[m][:, contrast == 0].mean()
    assert abs(diff - expect) < 1e-12

test_diff()
print("All checks passed")

### Exercise: Bug hunt: the vanishing baseline

This function should subtract each row's baseline (the mean of its first two bins) and return a new array, leaving the input alone. It changes the input instead. Fix it.

In [ ]:
import numpy as np

def baseline_subtract(x):
    out = x
    base = x[:, :2].mean(axis=1, keepdims=True)
    out -= base
    return out

raw = np.arange(12.0).reshape(3, 4)
print(baseline_subtract(raw)); print(raw)

In [ ]:
import numpy as np
def test_original_untouched():
    "raw is not changed by baseline_subtract"
    raw = np.arange(12.0).reshape(3, 4)
    before = raw.copy()
    baseline_subtract(raw)
    assert np.array_equal(raw, before), "The function changed its input."

def test_result():
    "each row has its first-two-bin mean subtracted"
    raw = np.arange(12.0).reshape(3, 4)
    out = baseline_subtract(raw)
    assert np.allclose(out, raw - raw[:, :2].mean(axis=1, keepdims=True))

test_original_untouched()
test_result()
print("All checks passed")